# The Baum–Welch Algorithm

**Companion wiki page:** https://ml-viz-ruby.vercel.app/wiki/baum-welch

A from-scratch NumPy implementation of Baum–Welch — EM for Hidden Markov Models. We build the forward pass, the backward pass, the responsibilities γ and ξ, and the M-step re-estimation, then watch the log-likelihood climb.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2d3148'
plt.rcParams['grid.color'] = '#2d3148'
np.random.seed(42)

## The weather HMM

Hidden states {Rainy, Sunny}; observations {Walk, Shop, Clean}. We generate a sequence from a *true* HMM, then pretend we don't know the parameters and recover them with Baum–Welch.

In [ ]:
# True parameters (what Baum-Welch should approximately recover)
pi_true = np.array([0.6, 0.4])
A_true = np.array([[0.7, 0.3],
                   [0.4, 0.6]])
B_true = np.array([[0.1, 0.4, 0.5],   # Rainy:  Walk, Shop, Clean
                   [0.6, 0.3, 0.1]])  # Sunny

def sample_hmm(pi, A, B, T):
    z = np.zeros(T, dtype=int)
    x = np.zeros(T, dtype=int)
    z[0] = np.random.choice(2, p=pi)
    x[0] = np.random.choice(3, p=B[z[0]])
    for t in range(1, T):
        z[t] = np.random.choice(2, p=A[z[t-1]])
        x[t] = np.random.choice(3, p=B[z[t]])
    return z, x

z_true, obs = sample_hmm(pi_true, A_true, B_true, T=500)
print("first 20 observations:", obs[:20])

## Forward and backward passes

- Forward: $\alpha_t(i) = P(x_{1:t}, z_t = i)$
- Backward: $\beta_t(i) = P(x_{t+1:T} \mid z_t = i)$

We scale $\alpha_t$ at each step (dividing by its sum) to avoid underflow; the same scalers normalize $\beta_t$, and their logs accumulate the log-likelihood.

In [ ]:
def forward(pi, A, B, x):
    T, N = len(x), len(pi)
    alpha = np.zeros((T, N))
    scale = np.zeros(T)
    alpha[0] = pi * B[:, x[0]]
    scale[0] = alpha[0].sum()
    alpha[0] /= scale[0]
    for t in range(1, T):
        alpha[t] = (alpha[t-1] @ A) * B[:, x[t]]
        scale[t] = alpha[t].sum()
        alpha[t] /= scale[t]
    return alpha, scale

def backward(A, B, x, scale):
    T, N = len(x), A.shape[0]
    beta = np.zeros((T, N))
    beta[-1] = 1.0
    for t in range(T - 2, -1, -1):
        beta[t] = A @ (B[:, x[t+1]] * beta[t+1])
        beta[t] /= scale[t+1]
    return beta

alpha, scale = forward(pi_true, A_true, B_true, obs)
beta = backward(A_true, B_true, obs, scale)
loglik_true = np.log(scale).sum()
print(f"log-likelihood under true params: {loglik_true:.2f}")

## E-step: responsibilities

$$\gamma_t(i) = \frac{\alpha_t(i)\beta_t(i)}{\sum_k \alpha_t(k)\beta_t(k)} \qquad
\xi_t(i,j) \propto \alpha_t(i)\, A_{ij}\, b_j(x_{t+1})\, \beta_{t+1}(j)$$

In [ ]:
def e_step(pi, A, B, x):
    T, N = len(x), len(pi)
    alpha, scale = forward(pi, A, B, x)
    beta = backward(A, B, x, scale)

    gamma = alpha * beta
    gamma /= gamma.sum(axis=1, keepdims=True)

    xi = np.zeros((T - 1, N, N))
    for t in range(T - 1):
        m = alpha[t][:, None] * A * (B[:, x[t+1]] * beta[t+1])[None, :]
        xi[t] = m / m.sum()
    loglik = np.log(scale).sum()
    return gamma, xi, loglik

## M-step: re-estimate parameters

$$\hat\pi_i = \gamma_1(i) \qquad
\hat A_{ij} = \frac{\sum_t \xi_t(i,j)}{\sum_t \gamma_t(i)} \qquad
\hat b_j(k) = \frac{\sum_{t: x_t=k} \gamma_t(j)}{\sum_t \gamma_t(j)}$$

In [ ]:
def m_step(gamma, xi, x, n_obs):
    pi = gamma[0]
    A = xi.sum(axis=0) / gamma[:-1].sum(axis=0)[:, None]
    B = np.zeros((gamma.shape[1], n_obs))
    for k in range(n_obs):
        B[:, k] = gamma[x == k].sum(axis=0)
    B /= gamma.sum(axis=0)[:, None]
    return pi, A, B

def baum_welch(x, n_states, n_obs, n_iter=50, seed=0):
    rng = np.random.default_rng(seed)
    pi = rng.dirichlet(np.ones(n_states))
    A = rng.dirichlet(np.ones(n_states), size=n_states)
    B = rng.dirichlet(np.ones(n_obs), size=n_states)
    history = []
    for _ in range(n_iter):
        gamma, xi, loglik = e_step(pi, A, B, x)
        history.append(loglik)
        pi, A, B = m_step(gamma, xi, x, n_obs)
    return pi, A, B, history

pi_est, A_est, B_est, history = baum_welch(obs, n_states=2, n_obs=3)
print("estimated A:\n", A_est.round(2))
print("estimated B:\n", B_est.round(2))

## The EM guarantee, visualized

Each iteration provably increases the log-likelihood — to a *local* optimum. (State labels may come out swapped vs the truth: the likelihood is invariant to permuting hidden states.)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history, color='#6366f1', lw=2)
plt.axhline(loglik_true, color='#2dd4bf', ls='--', label='log-lik under true params')
plt.xlabel('EM iteration'); plt.ylabel('log-likelihood')
plt.title('Baum–Welch: monotonic likelihood ascent')
plt.legend(); plt.grid(alpha=0.3); plt.show()

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise — state occupancy from α and β

**Recap:** the responsibility $\gamma_t(i) = \dfrac{\alpha_t(i)\beta_t(i)}{\sum_k \alpha_t(k)\beta_t(k)}$ tells you the posterior probability of being in state $i$ at time $t$, given the whole sequence.

Compute γ for the first 3 timesteps using the `alpha` and `beta` arrays already in memory.

In [ ]:
def gamma_first3(alpha, beta):
    # TODO(you): combine alpha and beta, normalize each row,
    # and return the first 3 rows (shape (3, 2)).
    ...

g3 = gamma_first3(alpha, beta)
g3

In [ ]:
# Run me — passes silently when correct
expected = (alpha * beta)
expected = (expected / expected.sum(axis=1, keepdims=True))[:3]
assert g3 is not None and not isinstance(g3, type(Ellipsis)), "fill in the TODO first"
assert np.allclose(np.asarray(g3), expected, atol=1e-8), "rows should be alpha*beta normalized"
assert np.allclose(np.asarray(g3).sum(axis=1), 1.0), "each row must sum to 1"
print()

<details>
<summary>Solution</summary>

```python
def gamma_first3(alpha, beta):
    g = alpha * beta
    g = g / g.sum(axis=1, keepdims=True)
    return g[:3]
```
</details>